# Module 6.1: Supervised Fine-Tuning (SFT)

Welcome to the final phase! Our model has finished its massive Pre-Training loop (Module 5). It is now a **Base Model** (like Llama 3 8B Base). It perfectly understands languages, logic, and coding, but it doesn't know how to act like an assistant.

If you prompt it with: `"What is the capital of France?"`, it might respond with `"What is the capital of Germany?"` because it simply completes web-style text.

In this notebook, we look at **Supervised Fine-Tuning**, teaching our model purely how to chat using specialized Prompt Architectures.

## 1. Chat Templates (The Secret Code)

To make the model behave, we construct a strict format of hidden syntax tokens. The model learns that when it sees a specific syntax, it is expected to answer rather than just autocomplete.

A popular format is **ChatML** (introduced by OpenAI and used by models like Qwen). It uses special control tokens:
1. `<|im_start|>`: Marks the beginning of a role (`im` = "instant message").
2. `<|im_end|>`: Marks the end of a message.
3. The role name (`system` / `user` / `assistant`) follows `<|im_start|>` to say who is speaking.

Different model families use different tokens. Llama 3, for example, does **not** use ChatML, it uses its own tokens like `<|start_header_id|>` and `<|eot_id|>`. The idea is the same; only the exact tokens differ.

Let's see what user messages ACTUALLY look like before they enter the model.

In [ ]:
import torch
torch.manual_seed(0)

def apply_chat_template(user_message, system_message="You are a helpful assistant."):
    """
    Converts human-readable messages into the strict ChatML formatting 
    required for a Fine-Tuned model.
    """
    formatted_prompt = (
        f"<|im_start|>system\n{system_message}<|im_end|>\n"
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n" # We leave this open so the model completes it!
    )
    return formatted_prompt

# Let's test it!
raw_input = "What is the capital of France?"
sft_input = apply_chat_template(raw_input)

print("--- RAW INPUT ---")
print(raw_input)
print("\n--- WHAT THE MODEL ACTUALLY SEES ---")
print(sft_input)

## 2. Training the SFT Model (Masking the Context)

During Supervised Fine-Tuning, we create a dataset of thousands of conversational examples. We pass the *entire* conversation (system + user + the perfect assistant reply) into the model using the Cross-Entropy Loss from Module 5.

**HOWEVER (The Golden Rule)**: We only calculate loss on the *assistant's* tokens! 
If we trained the model to predict the user's tokens too, it would start trying to make up what the user says next, which ruins generation. So we mask everything up to and including the `<|im_start|>assistant\n` header, and keep only the assistant's actual reply.

We do this masking with a special label value, `-100`. PyTorch's `CrossEntropyLoss` has an `ignore_index` argument (which defaults to exactly `-100`): any position whose label equals that value is skipped entirely when computing the loss. Every other label, including token id `0`, is trained normally. So `-100` is just "don't train here," nothing more.

Let's mark which positions of our ChatML prompt are masked (the system + user turn) versus kept (the assistant's reply).

In [ ]:
# Let's build a tiny conversation and show the mask boundary at the assistant turn.
# We pretend each "word" below is one token (real tokenizers split finer, but the idea holds).
ignore_index = -100  # CrossEntropyLoss(ignore_index=-100) skips these positions

prompt_part   = ["<|im_start|>", "user", "What", "is", "2+2?", "<|im_end|>", "<|im_start|>", "assistant"]
response_part = ["The", "answer", "is", "4", "<|im_end|>"]

tokens = prompt_part + response_part

# Pretend token ids (note: id 0 appears in the response and IS still trained!)
token_ids = [10, 11, 20, 21, 22, 12, 10, 13,   30, 31, 21, 0, 12]

# Mask EVERY prompt position with -100; keep the assistant's real reply.
labels = [ignore_index] * len(prompt_part) + token_ids[len(prompt_part):]

print(f"{'position':>8}  {'token':<14}{'id':>4}  {'label':>6}  trained?")
for i, (tok, tid, lab) in enumerate(zip(tokens, token_ids, labels)):
    trained = "no (masked)" if lab == ignore_index else "YES"
    print(f"{i:>8}  {tok:<14}{tid:>4}  {lab:>6}  {trained}")

print("\n-> Loss is computed ONLY on the assistant's reply. The -100 positions are skipped.")
print("-> Note position 11 has token id 0 but is still trained: 0 is a normal id, only -100 is ignored.")

# In practice, labels are the NEXT tokens (shifted left by one) so the model learns to predict
# the following token at each step. We omit that shift here to keep the masking idea clear.

## Summary

Supervised Fine-Tuning forces the model to adhere to a strict Question/Answer format. 

However, this doesn't prevent the model from being rude, saying "I don't know" immediately, or producing unsafe content. SFT just teaches it the format, not *values*. 

To instill human values, we move to the final piece of the modern AI pipeline: **Module 6.2: DPO (Preference Alignment)**!

### 🏋️ Try it yourself

1. **Multi-turn masking.** Extend the example to a two-turn conversation (user, assistant, user, assistant). Mask *both* user turns and *both* assistant headers with `-100`, keeping only the two assistant replies. Print the table and confirm only assistant replies are trained.
2. **Real loss with `ignore_index`.** Make up some fake `logits` of shape `(seq_len, vocab_size)` and a `labels` tensor that uses `-100` for masked positions, then call `torch.nn.functional.cross_entropy(logits, labels, ignore_index=-100)`. Change one masked label to a real token id and watch the loss change, proving the masked positions really are skipped.

In [ ]:
# Your code here!
# Hint for task 2:
# import torch
# import torch.nn.functional as F
# torch.manual_seed(0)
# logits = torch.randn(5, 100)            # 5 positions, vocab of 100
# labels = torch.tensor([-100, -100, 7, 42, 13])
# print(F.cross_entropy(logits, labels, ignore_index=-100))